<a href="https://colab.research.google.com/github/charre2021/polars_work_on_cyber_attacks/blob/main/Cyber_Attacks_Financial_And_Market_Impact.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cpi

In [128]:
import polars as pl
import polars.selectors as cs
import kagglehub
import requests
import os

In [4]:
path = kagglehub.dataset_download("aryanmdev/cyber-attacks-financial-and-market-impact")

100%|██████████| 169k/169k [00:00<00:00, 53.5MB/s]

Extracting files...


In [14]:
files_list = os.listdir(path)
market_impact_path = os.path.join(path, files_list[0])
financial_impact_path = os.path.join(path, files_list[1])
incidents_master_path = os.path.join(path, files_list[2])

In [15]:
market_impact_raw = pl.read_csv(market_impact_path)

In [17]:
financial_impact_raw = pl.read_csv(financial_impact_path)

In [19]:
incidents_master_raw = pl.read_csv(incidents_master_path)

In [ ]:
financial_impact_raw.filter(pl.col("notes") != "null").select(pl.col("notes"))

In [ ]:
financial_impact_raw.filter(pl.col("created_at") != pl.col("created_at").first()).select(pl.col("created_at"))

In [ ]:
financial_impact_raw.filter(pl.col("updated_at") != pl.col("updated_at").first()).select(pl.col("updated_at"))

In [37]:
fi_remove_last_two_cols = financial_impact_raw.drop(["created_at","updated_at"])

In [43]:
fi_remove_last_two_cols.head()

incident_id,direct_loss_usd,direct_loss_method,ransom_demanded_usd,ransom_paid_usd,ransom_source,recovery_cost_usd,legal_fees_usd,regulatory_fine_usd,insurance_payout_usd,total_loss_usd,total_loss_method,total_loss_lower_bound,total_loss_upper_bound,inflation_adjusted_usd,cpi_index_used,notes
str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str
"""2021-0508-001""",1.26e7,"""disclosed""",1.3803e7,null,null,9.4554e6,2.4965e6,90695.25,6.7563e6,2.4643e7,"""calculated""",1.5348e7,4.3747e7,2.9238e7,"""CPI-U 2021 (270.97)""",null
"""2025-1211-001""",7.6405e6,"""disclosed""",null,null,null,5.8572e6,1.8092e6,null,2.6910e6,1.5307e7,"""disclosed""",1.0206e7,1.8906e7,1.5307e7,"""CPI-U 2025 (321.5)""",null
"""2023-0115-001""",3.4882e7,"""calculated""",null,null,null,2.6404e7,1.0331e7,null,3.1760e7,7.1616e7,"""disclosed""",6.0854e7,1.0515e8,7.5565e7,"""CPI-U 2023 (304.702)""",null
"""2021-0315-001""",4.6822e6,"""disclosed""",null,null,null,3.6429e6,1.0290e6,null,1.7725e6,9354133.8,"""disclosed""",7.6490e6,1.4525e7,1.1098e7,"""CPI-U 2021 (270.97)""",null
"""2021-1204-001""",2.6846e6,"""estimated""",null,null,null,2.5749e6,206822.23,null,null,5.4663e6,"""estimated""",3.5198e6,6.7558e6,6.4856e6,"""CPI-U 2021 (270.97)""",null


In [71]:
fi_add_date = fi_remove_last_two_cols.with_columns(pl.col("incident_id").str.extract(r"(\d{4}-\d{4})", 1).str.to_date("%Y-%m%d").alias("incident_date"))

In [149]:
CPI_U_2026 = 326.588

fi_add_year_and_cpi = fi_add_date.with_columns(
    (pl.col("cpi_index_used").str.extract(r"\s{1}(\d{4})", 1).cast(pl.Int32).alias("cpi_base_year")),
    (pl.col("cpi_index_used").str.extract(r"\((\d+\.\d+)\)", 1).cast(pl.Float64).alias("base_cpi")),
    )

fi_add_new_cpi = fi_add_year_and_cpi.with_columns(pl.lit(CPI_U_2026).alias("cpi_u_2026"))

In [150]:
fi_add_new_cpi

incident_id,direct_loss_usd,direct_loss_method,ransom_demanded_usd,ransom_paid_usd,ransom_source,recovery_cost_usd,legal_fees_usd,regulatory_fine_usd,insurance_payout_usd,total_loss_usd,total_loss_method,total_loss_lower_bound,total_loss_upper_bound,inflation_adjusted_usd,cpi_index_used,notes,incident_date,cpi_base_year,base_cpi,cpi_u_2026
str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str,date,i32,f64,f64
"""2021-0508-001""",1.26e7,"""disclosed""",1.3803e7,null,null,9.4554e6,2.4965e6,90695.25,6.7563e6,2.4643e7,"""calculated""",1.5348e7,4.3747e7,2.9238e7,"""CPI-U 2021 (270.97)""",null,2021-05-08,2021,270.97,326.588
"""2025-1211-001""",7.6405e6,"""disclosed""",null,null,null,5.8572e6,1.8092e6,null,2.6910e6,1.5307e7,"""disclosed""",1.0206e7,1.8906e7,1.5307e7,"""CPI-U 2025 (321.5)""",null,2025-12-11,2025,321.5,326.588
"""2023-0115-001""",3.4882e7,"""calculated""",null,null,null,2.6404e7,1.0331e7,null,3.1760e7,7.1616e7,"""disclosed""",6.0854e7,1.0515e8,7.5565e7,"""CPI-U 2023 (304.702)""",null,2023-01-15,2023,304.702,326.588
"""2021-0315-001""",4.6822e6,"""disclosed""",null,null,null,3.6429e6,1.0290e6,null,1.7725e6,9354133.8,"""disclosed""",7.6490e6,1.4525e7,1.1098e7,"""CPI-U 2021 (270.97)""",null,2021-03-15,2021,270.97,326.588
"""2021-1204-001""",2.6846e6,"""estimated""",null,null,null,2.5749e6,206822.23,null,null,5.4663e6,"""estimated""",3.5198e6,6.7558e6,6.4856e6,"""CPI-U 2021 (270.97)""",null,2021-12-04,2021,270.97,326.588
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2022-0508-002""",3.0337e6,"""disclosed""",null,null,null,2.9023e6,674141.76,null,3.3616e6,6610099.6,"""disclosed""",4.7222e6,9.5480e6,7.2616e6,"""CPI-U 2022 (292.655)""",null,2022-05-08,2022,292.655,326.588
"""2023-0310-002""",2.4538e7,"""estimated""",7.5e7,5.1884e7,"""FBI disclosure""",1.8670e7,5.1836e6,null,null,1.0028e8,"""estimated""",7.1162e7,1.6302e8,1.0580e8,"""CPI-U 2023 (304.702)""",null,2023-03-10,2023,304.702,326.588
"""2025-1112-001""",740926.8,"""estimated""",null,null,null,243853.82,44814.27,null,601306.07,1.0296e6,"""disclosed""",780945.58,1.5010e6,1.0296e6,"""CPI-U 2025 (321.5)""",null,2025-11-12,2025,321.5,326.588
